# Practical Human Resource Allocation in Software Projects using Genetic Algorithm

## Introduction

Efficient allocation of human resources is one of the most critical and challenging problems in software project management. Software projects are constrained not only by task dependencies and deadlines, but also by human factors such as developer skills, experience levels, multitasking limitations, and knowledge continuity between tasks. Poor allocation decisions often lead to delays, excessive task switching, productivity loss, and increased project risk.

This project is a **replication and experimental reimplementation** of the research work entitled:

> **“Practical Human Resource Allocation in Software Projects Using Genetic Algorithms”**  
> Jihun Park, Dongwon Seo, Gwangui Hong, Donghwan Shin, Jimin Hwa, and Doo-Hwan Bae

The main objective of this project is to **develop a replica of the original approach proposed in the article and test it through our own implementation**. We do not aim to improve the original model, but rather to understand it better by rebuilding it and observing its behavior through experiments.

In this project, the software development process is modeled as an optimization problem. Each task has a workload and may depend on other tasks. Each developer has different skills and a specific experience level. A **Genetic Algorithm (GA)** is used to search for good task–developer assignments among many possible solutions.

To evaluate each solution, we simulate the execution of the project step by step. The simulator takes into account:
- Task dependencies
- Developer skills and levels
- The negative effect of multitasking
- The loss of efficiency when too many developers work on the same task

As in the original article, the quality of a solution is measured using four criteria:
- **CM (Completion Metric)**: how fast the project finishes compared to the best possible theoretical time (critical path lower bound).
- **CE (Concentration Efficiency)**: how well developers stay focused instead of multitasking, penalizes excessive multitasking and promotes developer focus.
- **CC (Continuity Criterion)**: encourages continuity of developers across dependent tasks to reduce knowledge loss.
- **BA (Balance Allocation)**: measures how evenly work is distributed between different staff levels

By running different experiments, we compare multi-objective optimization (CaseAll) with time-only optimization (CaseTime). The results show that considering human factors leads to more realistic and balanced schedules, which confirms the main ideas presented in the original article.


## Modeling the Input Data Sets

We modeled the project inputs in a simple and structured way. The input data represents a software project tasks and the people working on it. In our code, this is done using two main elements: **developers** and **tasks**.

### Developers

Each developer is modeled with three attributes:
- **ID**: a unique identifier (e.g. `d1`, `d2`)
- **Abilities**: a dictionary that defines the developer’s competence for each task type  
  (`analysis`, `design`, `implementation`, `testing`)
- **Level**: the experience level of the developer  
  (`junior`, `engineer`, `manager`, `director`)

The developer’s level is used to scale their abilities through a level factor. This reflects the idea that more experienced developers are generally more efficient, even when performing the same type of task.

### Tasks

Each task is defined by:
- **ID**: a unique task name (e.g. `t1`, `t2`)
- **Type**: the nature of the task (`analysis`, `design`, `implementation`, `testing`)
- **Effort**: the total amount of work required to complete the task
- **Dependencies**: a list of tasks that must be completed before this task can start

This structure allows us to model realistic project workflows with dependencies between tasks.

### Input Data Sets

We created **three different input data sets** to test the algorithm under different conditions:

- **Set 1**:  
  A small project with a limited number of developers and tasks.  
  This set is mainly used to validate the accuracy of the model and the algorithm.
  
- **Set 2**:  
  The same task structure as Set 1, but with additional developers.  
  This allows us to study how increasing team size affects project duration and multitasking.

- **Set 3**: 
  A larger project with more tasks, more dependencies, and more developers.  
This set is used to observe how the algorithm behaves when the project becomes bigger and more complex.

### Algorithm


#### 1. Chromosome Representation

A chromosome represents one possible project plan:

{"t1": ["d1", "d3"], "t2": ["d2"], ...}

Each chromosome fully defines how developers are allocated to all tasks.

---

#### 2. Project Simulation (`simulate`)
The function `simulate()` evaluates a chromosome by simulating the project execution step by step:
- A task can start only when all its prerequisites are completed
- Developers may work on multiple tasks at the same time
- Developer efficiency is reduced when multitasking
- A coordination penalty is applied when many developers work on the same task

The simulation returns the total project duration and developer workload statistics.

---

#### 3. Fitness Evaluation (`scores`)
Each solution is evaluated using four criteria:
- **CM (Cost Metric)**: compares project duration to an optimistic lower bound
- **CE (Concentration Efficiency)**: penalizes multitasking
- **CC (Continuity Consideration)**: rewards using the same developers across dependent tasks
- **BA (Balance Allocation)**: measures how evenly work is distributed across staff levels

The final fitness is a weighted sum of these four metrics.

---

### Genetic Algorithm Workflow

The main GA logic is implemented in the function `ga_set1()`.

#### Key GA Operations
- **Initialization (`init_pop`)**: generates a random population of solutions
- **Selection (`select`)**: chooses parents using tournament selection
- **Crossover (`crossover`)**: mixes assignments from two parents
- **Mutation (`mutate`)**: randomly modifies assignments to maintain diversity
- **Elitism**: preserves the best solutions across generations

---


### Genetic Algorithm Pseudocode

```text
Initialize population with random chromosomes
Evaluate fitness of each chromosome

FOR each generation:
    Keep best individuals (elitism)
    WHILE population not full:
        Select two parents
        Apply crossover
        Apply mutation
        Evaluate new chromosome
        Add to new population
    Update best solution found
END FOR

Return best chromosome


BEGIN GENETIC_ALGORITHM

1. INITIALIZATION
   Create an empty population P

   FOR i = 1 TO POP_SIZE:
       Create a random chromosome C
       Sanitize C (remove duplicates, enforce max developers per task)
       Add C to population P
   END FOR

   FOR each chromosome C in P:
       Evaluate fitness(C)
   END FOR

   Store the best chromosome found so far

2. MAIN LOOP
   FOR generation = 1 TO MAX_GENERATIONS:

       Sort population P by fitness (best first)

       Create a new empty population NEW_P

       -- ELITISM --
       FOR i = 1 TO ELITISM_COUNT:
           Copy P[i] into NEW_P
       END FOR

       -- CREATE NEW OFFSPRING --
       WHILE size(NEW_P) < POP_SIZE:

           -- SELECTION --
           parent1 = TournamentSelection(P)
           parent2 = TournamentSelection(P)

           -- CROSSOVER --
           child = Crossover(parent1, parent2)

           -- MUTATION --
           child = Mutate(child)

           -- SANITIZE --
           child = Sanitize(child)

           -- FITNESS EVALUATION --
           Evaluate fitness(child)

           Add child to NEW_P

       END WHILE

       -- REPLACE OLD POPULATION --
       P = NEW_P

       -- UPDATE BEST SOLUTION --
       IF best chromosome in P is better than global best:
           update global best
       END IF

   END FOR

3. RETURN global best chromosome

END GENETIC_ALGORITHM


FUNCTION Sanitize(chromosome, dev_list, MAX_PER_TASK):

    FOR each task T in chromosome:
        
        Create empty list CLEAN
        Create empty set SEEN

        FOR each developer D in chromosome[T]:
            IF D is in dev_list AND D not in SEEN:
                Add D to CLEAN
                Add D to SEEN
            END IF
        END FOR

        IF CLEAN is empty:
            Choose a random developer R from dev_list
            CLEAN = [R]
        END IF

        IF size(CLEAN) > MAX_PER_TASK:
            CLEAN = first MAX_PER_TASK elements of CLEAN
        END IF

        chromosome[T] = CLEAN

    END FOR

    RETURN chromosome
END FUNCTION


FUNCTION Crossover(parent1, parent2):

    Create empty chromosome CHILD

    FOR each task T:
        IF random() < 0.5:
            CHILD[T] = copy of parent1[T]
        ELSE:
            CHILD[T] = copy of parent2[T]
        END IF
    END FOR

    RETURN CHILD
END FUNCTION


FUNCTION Mutate(chromosome, dev_list, MUT_RATE, MAX_PER_TASK):

    FOR each task T in chromosome:

        IF random() < MUT_RATE:

            Choose random operation OP from {add, remove, replace}

            IF OP = add AND size(chromosome[T]) < MAX_PER_TASK:
                Choose developer D not already in chromosome[T]
                Add D to chromosome[T]

            ELSE IF OP = remove AND size(chromosome[T]) > 1:
                Remove a random developer from chromosome[T]

            ELSE IF OP = replace:
                Choose a random position i
                Choose a developer D not already in chromosome[T]
                Replace chromosome[T][i] with D

            END IF

        END IF

    END FOR

    RETURN Sanitize(chromosome)
END FUNCTION


FUNCTION TournamentSelection(population, fitnesses, K):

    Select K random indices from population → CANDIDATES

    BEST = first candidate

    FOR each candidate C in CANDIDATES:
        IF fitnesses[C] > fitnesses[BEST]:
            BEST = C
        END IF
    END FOR

    RETURN copy of population[BEST]
END FUNCTION


5. FITNESS EVALUATION (CM, CE, CC, BA)

FUNCTION Evaluate(chromosome):

    Sanitize(chromosome)

    SIM = SimulateSchedule(chromosome)

    CM = compute_CM(SIM)
    CE = compute_CE(SIM)
    CC = compute_CC(chromosome)
    BA = compute_BA(chromosome)

    FITNESS = w1*CM + w2*CE + w3*CC + w4*BA

    RETURN FITNESS
END FUNCTION


FUNCTION SimulateSchedule(chromosome):

    Initialize remaining effort for each task
    Initialize developer workloads
    time = 0

    WHILE not all tasks completed:

        enabled_tasks = tasks whose prerequisites are completed

        FOR each developer:
            count how many enabled tasks they are assigned to

        FOR each enabled task:
            compute progress based on assigned developers
            reduce remaining effort

        Mark tasks as completed if effort <= 0

        time = time + 1

    END WHILE

    RETURN simulation statistics (finish time, multitasking, etc.)
END FUNCTION

## Test 1 : Data Set 1

# Developers 

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#2c3e50; color:white;">
      <th style="padding:8px; border:1px solid #ddd;">Developer</th>
      <th style="padding:8px; border:1px solid #ddd;">Role</th>
      <th style="padding:8px; border:1px solid #ddd;">Analysis</th>
      <th style="padding:8px; border:1px solid #ddd;">Design</th>
      <th style="padding:8px; border:1px solid #ddd;">Implementation</th>
      <th style="padding:8px; border:1px solid #ddd;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d1</td>
      <td style="padding:8px; border:1px solid #ddd; color:#c0392b;"><b>Director</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d2</td>
      <td style="padding:8px; border:1px solid #ddd; color:#c0392b;"><b>Director</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.25</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d3</td>
      <td style="padding:8px; border:1px solid #ddd; color:#2980b9;"><b>Manager</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d4</td>
      <td style="padding:8px; border:1px solid #ddd; color:#2980b9;"><b>Manager</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d5</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td style="padding:8px; border:1px solid #ddd;">d6</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td style="padding:8px; border:1px solid #ddd;">d7</td>
      <td style="padding:8px; border:1px solid #ddd; color:#27ae60;"><b>Engineer</b></td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
      <td style="padding:8px; border:1px solid #ddd;">1.00</td>
      <td style="padding:8px; border:1px solid #ddd;">0.75</td>
    </tr>
  </tbody>
</table>


# Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ddd;">Task</th>
      <th style="padding:8px; border:1px solid #ddd;">Type</th>
      <th style="padding:8px; border:1px solid #ddd;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ddd;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#fdebd0;">
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Analysis</b></td>
      <td style="padding:8px; border:1px solid #ddd;">400</td>
      <td style="padding:8px; border:1px solid #ddd;">—</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t2</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">320</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t3</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td style="padding:8px; border:1px solid #ddd;">t4</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Design</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t1</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t5</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">240</td>
      <td style="padding:8px; border:1px solid #ddd;">t2</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t6</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">600</td>
      <td style="padding:8px; border:1px solid #ddd;">t3</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td style="padding:8px; border:1px solid #ddd;">t7</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Implementation</b></td>
      <td style="padding:8px; border:1px solid #ddd;">160</td>
      <td style="padding:8px; border:1px solid #ddd;">t4</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t8</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">100</td>
      <td style="padding:8px; border:1px solid #ddd;">t5</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t9</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">80</td>
      <td style="padding:8px; border:1px solid #ddd;">t6</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td style="padding:8px; border:1px solid #ddd;">t10</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">70</td>
      <td style="padding:8px; border:1px solid #ddd;">t7</td>
    </tr>
    <tr style="background-color:#f5b7b1;">
      <td style="padding:8px; border:1px solid #ddd;">t11</td>
      <td style="padding:8px; border:1px solid #ddd;"><b>Testing</b></td>
      <td style="padding:8px; border:1px solid #ddd;">80</td>
      <td style="padding:8px; border:1px solid #ddd;">t8, t9, t10</td>
    </tr>
  </tbody>
</table>


In [23]:
import random, math
from collections import deque

# Fixer la graine/seed aléatoire pour garantir la reproductibilité
random.seed(42)

# Facteur qui modélise l’expérience/seniorité de developpeur comme dans l’article
LEVEL_FACTOR = {
    "junior": 0.8,
    "engineer": 1.0,
    "manager": 1.1,
    "director": 1.2
}
# Structures de données d'entrée
class Developer:
    def __init__(self, id, abilities, level):
        self.id = id
        self.abilities = abilities
        self.level = level
        
class Task:
    
    def __init__(self, id, typ, effort, prereq):
        self.id = id
        self.typ = typ               #type de tâche (analysis, design, ...)
        self.effort = effort         #charge de travail
        self.prereq = list(prereq)   #liste des tâches prérequis

# Jeu de données de test 1 – Set 1
def generate_set1_data():
    
    devs = [
        Developer("d1", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25}, "director"),
        Developer("d2", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75}, "director"),
        Developer("d3", {"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0}, "manager"),
        Developer("d4", {"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75}, "manager"),
        Developer("d5", {"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0}, "engineer"),
        Developer("d6", {"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75}, "engineer"),
        Developer("d7", {"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75}, "engineer"),
    ]
    
    tasks = {
        "t1": Task("t1","analysis",400,[]),
        "t2": Task("t2","design",320,["t1"]),
        "t3": Task("t3","design",240,["t1"]),
        "t4": Task("t4","design",240,["t1"]),
        "t5": Task("t5","implementation",240,["t2"]),
        "t6": Task("t6","implementation",600,["t3"]),
        "t7": Task("t7","implementation",160,["t4"]),
        "t8": Task("t8","testing",100,["t5"]),
        "t9": Task("t9","testing",80,["t6"]),
        "t10": Task("t10","testing",70,["t7"]),
        "t11": Task("t11","testing",80,["t8","t9","t10"]),
    }

    return tasks, devs

# Algorithme génétique principal

def ga_set1(tasks, devs, WEIGHTS,
            pop_size=20, generations=50,
            mut=0.03, tourn=3, elit=2,
            max_per_task=3, verbose=True):
    
    # Indexation pour accélérer les calculs: 

    tids = list(tasks)                          # Liste des IDs de tâches
    tidx = {t:i for i,t in enumerate(tids)}     # TaskID -> index
    nT = len(tids)

    dids = [d.id for d in devs]                 # Liste des IDs développeurs
    didx = {d:i for i,d in enumerate(dids)}     # DevID -> index
    nD = len(dids)
    
    # Graphe des dépendances et efforts
    prereq = [[] for _ in range(nT)]            # Prérequis par tâche
    adj = [[] for _ in range(nT)]                # Successeurs
    effort = [0.0]*nT                            # Effort des tâches
    
    for tid, t in tasks.items():
        i = tidx[tid]
        effort[i] = float(t.effort)
        prereq[i] = [tidx[p] for p in t.prereq]
        for p in prereq[i]:
            adj[p].append(i)

    # Matrice des vitesses (dev × tâche)
    speed = [[0.0]*nT for _ in range(nD)]
    for d in devs:
        di = didx[d.id]
        lf = LEVEL_FACTOR.get(d.level, 1.0)
        for tid, t in tasks.items():
            speed[di][tidx[tid]] = d.abilities.get(t.typ, 0.0) * lf
            
    # Pénalité de coordination (multi-développeurs)
    def coord(k):
        return 1.0 if k <= 1 else max(0.4, 1.0 - 0.12*(k-1))
        
    # Nettoyage d’un chromosome
    def sanitize(ch):   # il sert à supprimer des doublons, limiter le nombre de devs par tâche et garantir au moins 1 dev par tâche
        for tid in tids:
            seen, out = set(), []
            for d in ch.get(tid, []):
                if d in didx and d not in seen:
                    seen.add(d); out.append(d)
            ch[tid] = out[:max_per_task] if out else [random.choice(dids)]
        return ch
        
    # Calcul de chemin critique
    def minTS():
        best_dur = []
        for ti in range(nT):
            best_sp = max(speed[di][ti] for di in range(nD))
            best_dur.append(effort[ti]/best_sp if best_sp > 0 else float("inf"))
            
        indeg = [len(prereq[i]) for i in range(nT)]
        q = deque([i for i in range(nT) if indeg[i] == 0])
        dp = best_dur[:]
        seen = 0
        
        while q:
            u = q.popleft(); seen += 1
            for v in adj[u]:
                dp[v] = max(dp[v], dp[u] + best_dur[v])
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)

        return max(dp) if seen == nT else min(best_dur)*nT

    MIN_TS = minTS()
    
    # Simulation temporelle du projet
    
    def simulate(ch, max_ticks=20000):  # Simule l’exécution du projet dans le temps

        rem = effort[:]
        done = [False]*nT
        need = [len(prereq[i]) for i in range(nT)]
        enabled = [i for i in range(nT) if need[i] == 0 and rem[i] > 0]

        load = [0]*nD
        ticks_work = [0]*nD
        ticks_mult = [0]*nD

        t = 0; done_cnt = 0

        while done_cnt < nT and t < max_ticks:
            if not enabled:
                t += 1; continue

            # calcul multitâche
            for di in range(nD): load[di] = 0
            for ti in enabled:
                for d in ch[tids[ti]]:
                    di = didx.get(d)
                    if di is not None:
                        load[di] += 1

            # statistiques CE
            for di in range(nD):
                if load[di] > 0:
                    ticks_work[di] += 1
                    if load[di] > 1:
                        ticks_mult[di] += 1

            finished = []
            for ti in enabled:
                raw, cnt = 0.0, 0
                for d in ch[tids[ti]]:
                    di = didx.get(d)
                    if di is None: continue
                    raw += speed[di][ti] / (load[di] if load[di] else 1)
                    cnt += 1

                if cnt:
                    rem[ti] -= raw * coord(cnt)
                    if rem[ti] <= 0 and not done[ti]:
                        done[ti] = True
                        finished.append(ti)
                        done_cnt += 1

            if finished:
                fin = set(finished)
                enabled = [x for x in enabled if x not in fin]
                for u in finished:
                    for v in adj[u]:
                        need[v] -= 1
                        if need[v] == 0 and not done[v] and rem[v] > 0:
                            enabled.append(v)

            t += 1

        dev_stats = {
            dids[i]: {
                "ticks_with_work": ticks_work[i],
                "ticks_with_multitask": ticks_mult[i]
            } for i in range(nD)
        }

        return t, dev_stats

    def scores(ch):
        ch=sanitize(ch)
        finish, dev_stats = simulate(ch)

        cm = (0.0 if MIN_TS<=0 or math.isinf(MIN_TS) else MIN_TS/max(finish,MIN_TS))
        ce_vals=[]
        for d in dids:
            tw=dev_stats[d]["ticks_with_work"]
            tm=dev_stats[d]["ticks_with_multitask"]
            ce_vals.append(1.0 if tw==0 else max(0.0,1.0-tm/tw))
        ce = sum(ce_vals)/len(ce_vals)

        total=ok=0
        for tid,t in tasks.items():
            for p in t.prereq:
                total+=1
                if set(ch[tid]) & set(ch[p]): ok+=1
        cc = ok/total if total else 1.0

        levels=sorted(set(d.level for d in devs)); dev_level={d.id:d.level for d in devs}; n=nT
        ba_vals=[]
        for lvl in levels:
            nums=[]
            for tid in tids:
                cnt=sum(1 for d in ch[tid] if dev_level.get(d)==lvl)
                nums.append(cnt*tasks[tid].effort)
            denom=sum(nums)
            if denom==0: ba_vals.append(1.0); continue
            ps=[x/denom for x in nums if x>0]
            H=0.0
            for p in ps: H -= p*(math.log(p)/math.log(n))
            ba_vals.append(H)
        ba=sum(ba_vals)/len(ba_vals) if ba_vals else 0.0

        fit = WEIGHTS["cm"]*cm + WEIGHTS["ce"]*ce + WEIGHTS["cc"]*cc + WEIGHTS["ba"]*ba
        return fit, {"cm":cm,"ce":ce,"cc":cc,"ba":ba,"sim":{"finish_time":finish, "dev_stats":dev_stats}}

    def init_pop():
        pop=[]
        for _ in range(pop_size):
            ch={tid: random.sample(dids, random.randint(1,max_per_task)) for tid in tids}
            pop.append(sanitize(ch))
        return pop

    def select(pop, fits):
        idxs=random.sample(range(len(pop)), tourn)
        b=max(idxs, key=lambda i: fits[i])
        return {tid: pop[b][tid][:] for tid in tids}

    def crossover(p1,p2):
        return {tid: (p1[tid][:] if random.random()<0.5 else p2[tid][:]) for tid in tids}

    def mutate(ch):
        for tid in tids:
            if random.random()<mut:
                op=random.choice(["add","remove","replace"])
                if op=="add" and len(ch[tid])<max_per_task:
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid].append(random.choice(cand))
                elif op=="remove" and len(ch[tid])>1:
                    ch[tid].pop(random.randrange(len(ch[tid])))
                elif op=="replace":
                    i=random.randrange(len(ch[tid]))
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid][i]=random.choice(cand)
        return sanitize(ch)

    pop=init_pop()
    fits,dets=zip(*(scores(ch) for ch in pop))
    fits,dets=list(fits),list(dets)
    bi=max(range(len(pop)), key=lambda i: fits[i])
    best=(pop[bi], fits[bi], dets[bi])

    for g in range(generations):
        ranked=sorted(range(len(pop)), key=lambda i: fits[i], reverse=True)
        new_pop=[pop[i] for i in ranked[:elit]]
        new_fits=[fits[i] for i in ranked[:elit]]
        new_dets=[dets[i] for i in ranked[:elit]]
        while len(new_pop)<pop_size:
            c=mutate(crossover(select(pop,fits), select(pop,fits)))
            f,d=scores(c)
            new_pop.append(c); new_fits.append(f); new_dets.append(d)
        pop,fits,dets=new_pop,new_fits,new_dets
        bi=max(range(len(pop)), key=lambda i: fits[i])
        if fits[bi]>best[1]: best=(pop[bi], fits[bi], dets[bi])
        if verbose:
            print(f"Gen {g+1}/{generations} best fitness: {best[1]:.4f}")
    return best

def multitask_sum(best_details):
    return sum(v["ticks_with_multitask"] for v in best_details["sim"]["dev_stats"].values())

if __name__ == "__main__":
    tasks, devs = generate_set1_data()

    # ---- Run CaseAll
    W_ALL  = {"cm":0.25, "ce":0.25, "cc":0.25, "ba":0.25}
    chromA, fitA, detA = ga_set1(tasks, devs, W_ALL, verbose=True)
    timeA = detA["sim"]["finish_time"]
    multiA = multitask_sum(detA)

    # ---- Run CaseTime
    W_TIME = {"cm":1.0, "ce":0.0, "cc":0.0, "ba":0.0}
    chromT, fitT, detT = ga_set1(tasks, devs, W_TIME, verbose=True)
    timeT = detT["sim"]["finish_time"]
    multiT = multitask_sum(detT)

    # ---- Print table (exact format)
    print("=== COMPARISON TABLE ===")
    print(f"{'Set':<6} | {'Time(All)':>10} | {'Time(Time)':>10} | {'Multitask(All)':>15} | {'Multitask(Time)':>15}")
    print(f"{'Set1':<6} | {timeA:>10.2f} | {timeT:>10.2f} | {multiA:>15.2f} | {multiT:>15.2f}")

    # afficher la meilleure solution pour CaseAll
    print("\n=== BEST SOLUTION (CaseAll) ===")
    print("Best fitness:", round(fitA, 6))
    print("Components:", {k: round(v, 6) for k, v in detA.items() if k in ["cm","ce","cc","ba"]})

    # afficher l'allocation des tâches (task -> devs)
    print("Finish time (sim):", detA["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromA, key=lambda x: int(x[1:])):  # t1, t2, ...
        print(f"  {tid} -> {chromA[tid]}")

    # afficher la meilleure solution pour CaseTime
    print("\n=== BEST SOLUTION (CaseTime) ===")
    print("Best fitness:", round(fitT, 6))
    print("Finish time (sim):", detT["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromT, key=lambda x: int(x[1:])):
        print(f"  {tid} -> {chromT[tid]}")


Gen 1/50 best fitness: 0.8676
Gen 2/50 best fitness: 0.8676
Gen 3/50 best fitness: 0.8676
Gen 4/50 best fitness: 0.8676
Gen 5/50 best fitness: 0.8676
Gen 6/50 best fitness: 0.8676
Gen 7/50 best fitness: 0.8883
Gen 8/50 best fitness: 0.8883
Gen 9/50 best fitness: 0.8920
Gen 10/50 best fitness: 0.8920
Gen 11/50 best fitness: 0.8920
Gen 12/50 best fitness: 0.8943
Gen 13/50 best fitness: 0.9038
Gen 14/50 best fitness: 0.9038
Gen 15/50 best fitness: 0.9038
Gen 16/50 best fitness: 0.9038
Gen 17/50 best fitness: 0.9048
Gen 18/50 best fitness: 0.9048
Gen 19/50 best fitness: 0.9049
Gen 20/50 best fitness: 0.9049
Gen 21/50 best fitness: 0.9067
Gen 22/50 best fitness: 0.9068
Gen 23/50 best fitness: 0.9068
Gen 24/50 best fitness: 0.9068
Gen 25/50 best fitness: 0.9068
Gen 26/50 best fitness: 0.9068
Gen 27/50 best fitness: 0.9068
Gen 28/50 best fitness: 0.9068
Gen 29/50 best fitness: 0.9161
Gen 30/50 best fitness: 0.9213
Gen 31/50 best fitness: 0.9213
Gen 32/50 best fitness: 0.9213
Gen 33/50 best fi

## Interpretation of the Results

### 1. Comparison Between CaseAll and CaseTime

The comparison table summarizes the performance of the genetic algorithm on **Set1** under two different optimization strategies:

- **CaseAll**: all objective components are weighted equally  
  (CM, CE, CC, BA each have a weight of 0.25).
- **CaseTime**: only the project duration (CM) is optimized  
  (CM = 1.0, all other weights = 0).

| Metric | CaseAll | CaseTime |
|------|---------|----------|
| Finish time | 824 | 833 |
| Multitasking | 111 | 741 |

**Observations:**
- CaseAll finishes slightly faster than CaseTime (824 vs 833).
- CaseAll strongly reduces multitasking compared to CaseTime.
- CaseTime allows much more multitasking because it only tries to minimize time and ignores developer concentration.

---

### 2. Best Solution for CaseAll

The best solution found with **CaseAll** reaches a **fitness of 0.948845**, which is very close to the theoretical maximum of 1.

#### Fitness components:
- **CM = 1.0**  
  The achieved project duration is equal to the estimated critical path lower bound.
- **CE ≈ 0.98**  
  Developers are rarely multitasking, meaning good focus and low context switching.
- **CC = 1.0**  
  All precedence-related tasks share at least one common developer, ensuring strong continuity.
- **BA ≈ 0.82**  
  The workload is fairly well balanced across staff levels, though not perfectly uniform.

The final project duration is **824 time units**.

The task allocation shows that:
- Tasks are often assigned to small teams rather than a single developer.
- Predecessor tasks frequently share developers, which helps continuity.
- Multitasking is limited and controlled.

---

### 3. Best Solution for CaseTime

For **CaseTime**, the fitness is **1.0**, which is expected because the objective only considers project duration.

- The finish time is **833 time units**, slightly worse than CaseAll.
- Multitasking is much higher, as developers are assigned to many tasks at the same time.
- Continuity, balance, and concentration are not enforced, leading to heavier workloads on some developers.

This solution demonstrates that:
- Minimizing time alone can lead to inefficient or unrealistic work patterns.
- Ignoring human-related constraints results in excessive multitasking.

---

### 4. Global Interpretation

These results highlight the trade-off between:
- **Pure time optimization (CaseTime)**  
- **Balanced and realistic scheduling (CaseAll)**

While CaseTime focuses only on speed, CaseAll produces a schedule that is:
- Slightly faster,
- More realistic,
- Better aligned with human work constraints.

This confirms the importance of using multi-objective optimization rather than optimizing project duration alone.


## Test 2 : Data Set 2


# Developers

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#1f2d3d; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Developer</th>
      <th style="padding:8px; border:1px solid #ccc;">Role</th>
      <th style="padding:8px; border:1px solid #ccc;">Analysis</th>
      <th style="padding:8px; border:1px solid #ccc;">Design</th>
      <th style="padding:8px; border:1px solid #ccc;">Implementation</th>
      <th style="padding:8px; border:1px solid #ccc;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td>d1</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>1.25</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d2</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d3</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>0.75</td><td>1.00</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d4</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>1.00</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d5</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>0.75</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d6</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>0.75</td><td>1.00</td><td>0.75</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d7</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d8</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.90</td><td>0.90</td><td>0.90</td><td>0.90</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td>d9</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.80</td><td>0.80</td><td>0.80</td><td>0.80</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d10</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.85</td><td>0.85</td><td>0.85</td><td>0.85</td>
    </tr>
  </tbody>
</table>


# Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Task</th>
      <th style="padding:8px; border:1px solid #ccc;">Type</th>
      <th style="padding:8px; border:1px solid #ccc;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ccc;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#fdebd0;">
      <td>t1</td><td><b>Analysis</b></td><td>400</td><td>—</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t2</td><td><b>Design</b></td><td>320</td><td>t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t3</td><td><b>Design</b></td><td>240</td><td>t1</td>
    </tr>
    <tr style="background-color:#d6eaf8;">
      <td>t4</td><td><b>Design</b></td><td>240</td><td>t1</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t5</td><td><b>Implementation</b></td><td>240</td><td>t2</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t6</td><td><b>Implementation</b></td><td>600</td><td>t3</td>
    </tr>
    <tr style="background-color:#d5f5e3;">
      <td>t7</td><td><b>Implementation</b></td><td>160</td><td>t4</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t8</td><td><b>Testing</b></td><td>100</td><td>t5</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t9</td><td><b>Testing</b></td><td>80</td><td>t6</td>
    </tr>
    <tr style="background-color:#fadbd8;">
      <td>t10</td><td><b>Testing</b></td><td>70</td><td>t7</td>
    </tr>
    <tr style="background-color:#f5b7b1;">
      <td>t11</td><td><b>Testing</b></td><td>80</td><td>t8, t9, t10</td>
    </tr>
  </tbody>
</table>



In [11]:
import random, math
from collections import deque

random.seed(42)
LEVEL_FACTOR = {"junior":.8,"engineer":1.0,"manager":1.1,"director":1.2}

class Developer:
    def __init__(s,id,abilities,level): s.id=id; s.abilities=abilities; s.level=level
class Task:
    def __init__(s,id,typ,effort,prereq): s.id=id; s.typ=typ; s.effort=effort; s.prereq=list(prereq)

def generate_set2_data():
    # --- Developers (10 total) ---
    devs = [
        Developer("d1", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25}, "director"),
        Developer("d2", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75}, "director"),
        Developer("d3", {"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0}, "manager"),
        Developer("d4", {"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75}, "manager"),
        Developer("d5", {"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0}, "engineer"),
        Developer("d6", {"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75}, "engineer"),
        Developer("d7", {"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75}, "engineer"),
        Developer("d8", {"analysis":0.9,"design":0.9,"implementation":0.9,"testing":0.9}, "engineer"),
        Developer("d9", {"analysis":0.8,"design":0.8,"implementation":0.8,"testing":0.8}, "engineer"),
        Developer("d10",{"analysis":0.85,"design":0.85,"implementation":0.85,"testing":0.85}, "engineer"),
    ]
    # --- Tasks (same 11 tasks as Set1) ---
    tasks = {
        "t1": Task("t1","analysis",400,[]),
        "t2": Task("t2","design",320,["t1"]),
        "t3": Task("t3","design",240,["t1"]),
        "t4": Task("t4","design",240,["t1"]),
        "t5": Task("t5","implementation",240,["t2"]),
        "t6": Task("t6","implementation",600,["t3"]),
        "t7": Task("t7","implementation",160,["t4"]),
        "t8": Task("t8","testing",100,["t5"]),
        "t9": Task("t9","testing",80,["t6"]),
        "t10": Task("t10","testing",70,["t7"]),
        "t11": Task("t11","testing",80,["t8","t9","t10"]),
    }
    return tasks, devs

def ga_set2(tasks, devs, WEIGHTS, pop_size=20, generations=50, mut=0.03, tourn=3, elit=2, max_per_task=3, verbose=True):
    tids=list(tasks); tidx={t:i for i,t in enumerate(tids)}; nT=len(tids)
    dids=[d.id for d in devs]; didx={d:i for i,d in enumerate(dids)}; nD=len(dids)

    prereq=[[] for _ in range(nT)]; adj=[[] for _ in range(nT)]; effort=[0.0]*nT
    for tid,t in tasks.items():
        i=tidx[tid]; effort[i]=float(t.effort); prereq[i]=[tidx[p] for p in t.prereq]
        for p in prereq[i]: adj[p].append(i)

    speed=[[0.0]*nT for _ in range(nD)]
    for d in devs:
        di=didx[d.id]; lf=LEVEL_FACTOR.get(d.level,1.0)
        for tid,t in tasks.items():
            speed[di][tidx[tid]]=d.abilities.get(t.typ,0.0)*lf

    def coord(k): return 1.0 if k<=1 else max(0.4,1.0-0.12*(k-1))

    def sanitize(ch):
        for tid in tids:
            seen=set(); out=[]
            for d in ch.get(tid,[]):
                if d in didx and d not in seen:
                    seen.add(d); out.append(d)
            ch[tid]=out[:max_per_task] if out else [random.choice(dids)]
        return ch

    def minTS():
        best_dur=[]
        for ti in range(nT):
            best_sp=max(speed[di][ti] for di in range(nD))
            best_dur.append(effort[ti]/best_sp if best_sp>0 else float("inf"))
        indeg=[len(prereq[i]) for i in range(nT)]
        q=deque([i for i in range(nT) if indeg[i]==0]); dp=best_dur[:]; seen=0
        while q:
            u=q.popleft(); seen+=1
            for v in adj[u]:
                dp[v]=max(dp[v],dp[u]+best_dur[v])
                indeg[v]-=1
                if indeg[v]==0: q.append(v)
        return max(dp) if seen==nT else min(best_dur)*nT

    MIN_TS=minTS()

    def simulate(ch, max_ticks=20000):
        rem=effort[:]; done=[False]*nT; need=[len(prereq[i]) for i in range(nT)]
        enabled=[i for i in range(nT) if need[i]==0 and rem[i]>0]

        load=[0]*nD
        ticks_work=[0]*nD
        ticks_mult=[0]*nD

        t=0; done_cnt=0
        while done_cnt<nT and t<max_ticks:
            if not enabled:
                t+=1
                continue

            for di in range(nD): load[di]=0
            for ti in enabled:
                for d in ch[tids[ti]]:
                    di=didx.get(d)
                    if di is not None: load[di]+=1

            for di in range(nD):
                if load[di]>0:
                    ticks_work[di]+=1
                    if load[di]>1: ticks_mult[di]+=1

            finished=[]
            for ti in enabled:
                raw=0.0; cnt=0
                for d in ch[tids[ti]]:
                    di=didx.get(d)
                    if di is None: continue
                    denom=load[di] if load[di] else 1
                    raw += speed[di][ti]/denom
                    cnt += 1
                if cnt:
                    rem[ti] -= raw*coord(cnt)
                    if rem[ti]<=0 and not done[ti]:
                        done[ti]=True; finished.append(ti); done_cnt+=1

            if finished:
                fin=set(finished)
                enabled=[x for x in enabled if x not in fin]
                for u in finished:
                    for v in adj[u]:
                        need[v]-=1
                        if need[v]==0 and (not done[v]) and rem[v]>0:
                            enabled.append(v)
            t+=1

        dev_stats = {
            dids[i]: {"ticks_with_work": ticks_work[i], "ticks_with_multitask": ticks_mult[i]}
            for i in range(nD)
        }
        return t, dev_stats

    def scores(ch):
        ch=sanitize(ch)
        finish, dev_stats = simulate(ch)

        cm = (0.0 if MIN_TS<=0 or math.isinf(MIN_TS) else MIN_TS/max(finish,MIN_TS))
        ce_vals=[]
        for d in dids:
            tw=dev_stats[d]["ticks_with_work"]
            tm=dev_stats[d]["ticks_with_multitask"]
            ce_vals.append(1.0 if tw==0 else max(0.0,1.0-tm/tw))
        ce = sum(ce_vals)/len(ce_vals)

        total=ok=0
        for tid,t in tasks.items():
            for p in t.prereq:
                total+=1
                if set(ch[tid]) & set(ch[p]): ok+=1
        cc = ok/total if total else 1.0

        levels=sorted(set(d.level for d in devs)); dev_level={d.id:d.level for d in devs}; n=nT
        ba_vals=[]
        for lvl in levels:
            nums=[]
            for tid in tids:
                cnt=sum(1 for d in ch[tid] if dev_level.get(d)==lvl)
                nums.append(cnt*tasks[tid].effort)
            denom=sum(nums)
            if denom==0: ba_vals.append(1.0); continue
            ps=[x/denom for x in nums if x>0]
            H=0.0
            for p in ps: H -= p*(math.log(p)/math.log(n))
            ba_vals.append(H)
        ba=sum(ba_vals)/len(ba_vals) if ba_vals else 0.0

        fit = WEIGHTS["cm"]*cm + WEIGHTS["ce"]*ce + WEIGHTS["cc"]*cc + WEIGHTS["ba"]*ba
        return fit, {"cm":cm,"ce":ce,"cc":cc,"ba":ba,"sim":{"finish_time":finish, "dev_stats":dev_stats}}

    def init_pop():
        pop=[]
        for _ in range(pop_size):
            ch={tid: random.sample(dids, random.randint(1,max_per_task)) for tid in tids}
            pop.append(sanitize(ch))
        return pop

    def select(pop, fits):
        idxs=random.sample(range(len(pop)), tourn)
        b=max(idxs, key=lambda i: fits[i])
        return {tid: pop[b][tid][:] for tid in tids}

    def crossover(p1,p2):
        return {tid: (p1[tid][:] if random.random()<0.5 else p2[tid][:]) for tid in tids}

    def mutate(ch):
        for tid in tids:
            if random.random()<mut:
                op=random.choice(["add","remove","replace"])
                if op=="add" and len(ch[tid])<max_per_task:
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid].append(random.choice(cand))
                elif op=="remove" and len(ch[tid])>1:
                    ch[tid].pop(random.randrange(len(ch[tid])))
                elif op=="replace":
                    i=random.randrange(len(ch[tid]))
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid][i]=random.choice(cand)
        return sanitize(ch)

    pop=init_pop()
    fits,dets=zip(*(scores(ch) for ch in pop))
    fits,dets=list(fits),list(dets)
    bi=max(range(len(pop)), key=lambda i: fits[i])
    best=(pop[bi], fits[bi], dets[bi])

    for g in range(generations):
        ranked=sorted(range(len(pop)), key=lambda i: fits[i], reverse=True)
        new_pop=[pop[i] for i in ranked[:elit]]
        new_fits=[fits[i] for i in ranked[:elit]]
        new_dets=[dets[i] for i in ranked[:elit]]
        while len(new_pop)<pop_size:
            c=mutate(crossover(select(pop,fits), select(pop,fits)))
            f,d=scores(c)
            new_pop.append(c); new_fits.append(f); new_dets.append(d)
        pop,fits,dets=new_pop,new_fits,new_dets
        bi=max(range(len(pop)), key=lambda i: fits[i])
        if fits[bi]>best[1]: best=(pop[bi], fits[bi], dets[bi])
        if verbose:
            print(f"Gen {g+1}/{generations} best fitness: {best[1]:.4f}")
    return best

def multitask_sum(best_details):
    return sum(v["ticks_with_multitask"] for v in best_details["sim"]["dev_stats"].values())

if __name__ == "__main__":
    tasks, devs = generate_set2_data()
    
    # ---- Run CaseAll
    W_ALL  = {"cm":0.25, "ce":0.25, "cc":0.25, "ba":0.25}
    chromA, fitA, detA = ga_set2(tasks, devs, W_ALL, verbose=True)
    timeA = detA["sim"]["finish_time"]
    multiA = multitask_sum(detA)

    # ---- Run CaseTime
    W_TIME = {"cm":1.0, "ce":0.0, "cc":0.0, "ba":0.0}
    chromT, fitT, detT = ga_set2(tasks, devs, W_TIME, verbose=True)
    timeT = detT["sim"]["finish_time"]
    multiT = multitask_sum(detT)

    # ---- Print table (exact format)
    print("=== COMPARISON TABLE ===")
    print(f"{'Set':<6} | {'Time(All)':>10} | {'Time(Time)':>10} | {'Multitask(All)':>15} | {'Multitask(Time)':>15}")
    print(f"{'Set2':<6} | {timeA:>10.2f} | {timeT:>10.2f} | {multiA:>15.2f} | {multiT:>15.2f}")

    # ---- Best solution (CaseAll)
    print("\n=== BEST SOLUTION (CaseAll) ===")
    print("Best fitness:", round(fitA, 6))
    print("Components:", {k: round(v, 6) for k, v in detA.items() if k in ["cm","ce","cc","ba"]})

    # afficher l'allocation des tâches (task -> devs)
    print("Finish time (sim):", detA["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromA, key=lambda x: int(x[1:])):  # t1, t2, ...
        print(f"  {tid} -> {chromA[tid]}")

    # afficher la meilleure solution pour CaseTime
    print("\n=== BEST SOLUTION (CaseTime) ===")
    print("Best fitness:", round(fitT, 6))
    print("Finish time (sim):", detT["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromT, key=lambda x: int(x[1:])):
        print(f"  {tid} -> {chromT[tid]}")


Gen 1/50 best fitness: 0.8055
Gen 2/50 best fitness: 0.8143
Gen 3/50 best fitness: 0.8181
Gen 4/50 best fitness: 0.8403
Gen 5/50 best fitness: 0.8603
Gen 6/50 best fitness: 0.8889
Gen 7/50 best fitness: 0.8889
Gen 8/50 best fitness: 0.8911
Gen 9/50 best fitness: 0.8965
Gen 10/50 best fitness: 0.8987
Gen 11/50 best fitness: 0.9028
Gen 12/50 best fitness: 0.9044
Gen 13/50 best fitness: 0.9044
Gen 14/50 best fitness: 0.9044
Gen 15/50 best fitness: 0.9091
Gen 16/50 best fitness: 0.9091
Gen 17/50 best fitness: 0.9108
Gen 18/50 best fitness: 0.9111
Gen 19/50 best fitness: 0.9111
Gen 20/50 best fitness: 0.9112
Gen 21/50 best fitness: 0.9135
Gen 22/50 best fitness: 0.9138
Gen 23/50 best fitness: 0.9138
Gen 24/50 best fitness: 0.9162
Gen 25/50 best fitness: 0.9180
Gen 26/50 best fitness: 0.9193
Gen 27/50 best fitness: 0.9233
Gen 28/50 best fitness: 0.9233
Gen 29/50 best fitness: 0.9234
Gen 30/50 best fitness: 0.9234
Gen 31/50 best fitness: 0.9254
Gen 32/50 best fitness: 0.9254
Gen 33/50 best fi

## Test 3 : Data Set 3

## Developers

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#1f2d3d; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Developer</th>
      <th style="padding:8px; border:1px solid #ccc;">Role</th>
      <th style="padding:8px; border:1px solid #ccc;">Analysis</th>
      <th style="padding:8px; border:1px solid #ccc;">Design</th>
      <th style="padding:8px; border:1px solid #ccc;">Implementation</th>
      <th style="padding:8px; border:1px solid #ccc;">Testing</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#ecf0f1;">
      <td>d1</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>1.25</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d2</td><td style="color:#c0392b;"><b>Director</b></td>
      <td>1.25</td><td>1.00</td><td>1.25</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d3</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>0.75</td><td>1.00</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d4</td><td style="color:#2980b9;"><b>Manager</b></td>
      <td>0.75</td><td>1.00</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d5</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>0.75</td><td>1.00</td>
    </tr>
    <tr style="background-color:#ffffff;">
      <td>d6</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>0.75</td><td>1.00</td><td>0.75</td><td>0.75</td>
    </tr>
    <tr style="background-color:#ecf0f1;">
      <td>d7</td><td style="color:#27ae60;"><b>Engineer</b></td>
      <td>1.00</td><td>0.75</td><td>1.00</td><td>0.75</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d8</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.90</td><td>0.90</td><td>0.90</td><td>0.90</td>
    </tr>
    <tr style="background-color:#fdfefe;">
      <td>d9</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.80</td><td>0.80</td><td>0.80</td><td>0.80</td>
    </tr>
    <tr style="background-color:#e8f8f5;">
      <td>d10</td><td style="color:#16a085;"><b>Engineer</b></td>
      <td>0.85</td><td>0.85</td><td>0.85</td><td>0.85</td>
    </tr>
  </tbody>
</table>


## Tasks

<table style="border-collapse: collapse; width:100%; font-family: Arial;">
  <thead>
    <tr style="background-color:#34495e; color:white;">
      <th style="padding:8px; border:1px solid #ccc;">Task</th>
      <th style="padding:8px; border:1px solid #ccc;">Type</th>
      <th style="padding:8px; border:1px solid #ccc;">Effort (h)</th>
      <th style="padding:8px; border:1px solid #ccc;">Dependencies</th>
    </tr>
  </thead>
  <tbody>
    <!-- Analysis -->
    <tr style="background-color:#fdebd0;"><td>t1</td><td><b>Analysis</b></td><td>400</td><td>—</td></tr>
    <tr style="background-color:#fdebd0;"><td>t12</td><td><b>Analysis</b></td><td>300</td><td>—</td></tr>
    <!-- Design -->
    <tr style="background-color:#d6eaf8;"><td>t2</td><td><b>Design</b></td><td>320</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t3</td><td><b>Design</b></td><td>240</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t4</td><td><b>Design</b></td><td>240</td><td>t1</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t13</td><td><b>Design</b></td><td>250</td><td>t12</td></tr>
    <tr style="background-color:#d6eaf8;"><td>t16</td><td><b>Design</b></td><td>200</td><td>t12</td></tr>
    <!-- Implementation -->
    <tr style="background-color:#d5f5e3;"><td>t5</td><td><b>Implementation</b></td><td>240</td><td>t2</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t6</td><td><b>Implementation</b></td><td>600</td><td>t3</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t7</td><td><b>Implementation</b></td><td>160</td><td>t4</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t14</td><td><b>Implementation</b></td><td>400</td><td>t13</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t17</td><td><b>Implementation</b></td><td>350</td><td>t16</td></tr>
    <tr style="background-color:#d5f5e3;"><td>t19</td><td><b>Implementation</b></td><td>300</td><td>t13</td></tr>
    <!-- Testing -->
    <tr style="background-color:#fadbd8;"><td>t8</td><td><b>Testing</b></td><td>100</td><td>t5</td></tr>
    <tr style="background-color:#fadbd8;"><td>t9</td><td><b>Testing</b></td><td>80</td><td>t6</td></tr>
    <tr style="background-color:#fadbd8;"><td>t10</td><td><b>Testing</b></td><td>70</td><td>t7</td></tr>
    <tr style="background-color:#fadbd8;"><td>t15</td><td><b>Testing</b></td><td>90</td><td>t14</td></tr>
    <tr style="background-color:#fadbd8;"><td>t18</td><td><b>Testing</b></td><td>85</td><td>t17</td></tr>
    <tr style="background-color:#fadbd8;"><td>t20</td><td><b>Testing</b></td><td>75</td><td>t19</td></tr>
    <!-- Final merge -->
    <tr style="background-color:#f5b7b1;">
      <td>t11</td><td><b>Testing</b></td><td>80</td><td>t8, t9, t10</td>
    </tr>
    <tr style="background-color:#f1948a;">
      <td>t21</td><td><b>Testing</b></td><td>95</td><td>t15, t18, t20</td>
    </tr>
  </tbody>
</table>


In [16]:
import random, math
from collections import deque

random.seed(42)
LEVEL_FACTOR = {"junior":.8,"engineer":1.0,"manager":1.1,"director":1.2}

class Developer:
    def __init__(s,id,abilities,level): s.id=id; s.abilities=abilities; s.level=level
class Task:
    def __init__(s,id,typ,effort,prereq): s.id=id; s.typ=typ; s.effort=effort; s.prereq=list(prereq)

def generate_set3_data():
    # --- Developers (defined independently) ---
    devs = [
        Developer("d1", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":1.25}, "director"),
        Developer("d2", {"analysis":1.25,"design":1.0,"implementation":1.25,"testing":0.75}, "director"),
        Developer("d3", {"analysis":0.75,"design":0.75,"implementation":1.0,"testing":1.0}, "manager"),
        Developer("d4", {"analysis":0.75,"design":1.0,"implementation":1.0,"testing":0.75}, "manager"),
        Developer("d5", {"analysis":1.0,"design":0.75,"implementation":0.75,"testing":1.0}, "engineer"),
        Developer("d6", {"analysis":0.75,"design":1.0,"implementation":0.75,"testing":0.75}, "engineer"),
        Developer("d7", {"analysis":1.0,"design":0.75,"implementation":1.0,"testing":0.75}, "engineer"),
        Developer("d8", {"analysis":0.9,"design":0.9,"implementation":0.9,"testing":0.9}, "engineer"),
        Developer("d9", {"analysis":0.8,"design":0.8,"implementation":0.8,"testing":0.8}, "engineer"),
        Developer("d10",{"analysis":0.85,"design":0.85,"implementation":0.85,"testing":0.85}, "engineer"),
    ]

    # --- Tasks (21 tasks total, fully defined) ---
    tasks = {
        "t1": Task("t1","analysis",400,[]),
        "t2": Task("t2","design",320,["t1"]),
        "t3": Task("t3","design",240,["t1"]),
        "t4": Task("t4","design",240,["t1"]),
        "t5": Task("t5","implementation",240,["t2"]),
        "t6": Task("t6","implementation",600,["t3"]),
        "t7": Task("t7","implementation",160,["t4"]),
        "t8": Task("t8","testing",100,["t5"]),
        "t9": Task("t9","testing",80,["t6"]),
        "t10": Task("t10","testing",70,["t7"]),
        "t11": Task("t11","testing",80,["t8","t9","t10"]),
        "t12": Task("t12","analysis",300,[]),
        "t13": Task("t13","design",250,["t12"]),
        "t14": Task("t14","implementation",400,["t13"]),
        "t15": Task("t15","testing",90,["t14"]),
        "t16": Task("t16","design",200,["t12"]),
        "t17": Task("t17","implementation",350,["t16"]),
        "t18": Task("t18","testing",85,["t17"]),
        "t19": Task("t19","implementation",300,["t13"]),
        "t20": Task("t20","testing",75,["t19"]),
        "t21": Task("t21","testing",95,["t15","t18","t20"]),
    }

    return tasks, devs


def ga_set3(tasks, devs, WEIGHTS, pop_size=20, generations=50, mut=0.03, tourn=3, elit=2, max_per_task=3, verbose=True):
    tids=list(tasks); tidx={t:i for i,t in enumerate(tids)}; nT=len(tids)
    dids=[d.id for d in devs]; didx={d:i for i,d in enumerate(dids)}; nD=len(dids)

    prereq=[[] for _ in range(nT)]; adj=[[] for _ in range(nT)]; effort=[0.0]*nT
    for tid,t in tasks.items():
        i=tidx[tid]; effort[i]=float(t.effort); prereq[i]=[tidx[p] for p in t.prereq]
        for p in prereq[i]: adj[p].append(i)

    speed=[[0.0]*nT for _ in range(nD)]
    for d in devs:
        di=didx[d.id]; lf=LEVEL_FACTOR.get(d.level,1.0)
        for tid,t in tasks.items():
            speed[di][tidx[tid]]=d.abilities.get(t.typ,0.0)*lf

    def coord(k): return 1.0 if k<=1 else max(0.4,1.0-0.12*(k-1))

    def sanitize(ch):
        for tid in tids:
            seen=set(); out=[]
            for d in ch.get(tid,[]):
                if d in didx and d not in seen:
                    seen.add(d); out.append(d)
            ch[tid]=out[:max_per_task] if out else [random.choice(dids)]
        return ch

    def minTS():
        best_dur=[]
        for ti in range(nT):
            best_sp=max(speed[di][ti] for di in range(nD))
            best_dur.append(effort[ti]/best_sp if best_sp>0 else float("inf"))
        indeg=[len(prereq[i]) for i in range(nT)]
        q=deque([i for i in range(nT) if indeg[i]==0]); dp=best_dur[:]; seen=0
        while q:
            u=q.popleft(); seen+=1
            for v in adj[u]:
                dp[v]=max(dp[v],dp[u]+best_dur[v])
                indeg[v]-=1
                if indeg[v]==0: q.append(v)
        return max(dp) if seen==nT else min(best_dur)*nT

    MIN_TS=minTS()

    def simulate(ch, max_ticks=20000):
        rem=effort[:]; done=[False]*nT; need=[len(prereq[i]) for i in range(nT)]
        enabled=[i for i in range(nT) if need[i]==0 and rem[i]>0]

        load=[0]*nD
        ticks_work=[0]*nD
        ticks_mult=[0]*nD

        t=0; done_cnt=0
        while done_cnt<nT and t<max_ticks:
            if not enabled:
                t+=1
                continue

            for di in range(nD): load[di]=0
            for ti in enabled:
                for d in ch[tids[ti]]:
                    di=didx.get(d)
                    if di is not None: load[di]+=1

            for di in range(nD):
                if load[di]>0:
                    ticks_work[di]+=1
                    if load[di]>1: ticks_mult[di]+=1

            finished=[]
            for ti in enabled:
                raw=0.0; cnt=0
                for d in ch[tids[ti]]:
                    di=didx.get(d)
                    if di is None: continue
                    denom=load[di] if load[di] else 1
                    raw += speed[di][ti]/denom
                    cnt += 1
                if cnt:
                    rem[ti] -= raw*coord(cnt)
                    if rem[ti]<=0 and not done[ti]:
                        done[ti]=True; finished.append(ti); done_cnt+=1

            if finished:
                fin=set(finished)
                enabled=[x for x in enabled if x not in fin]
                for u in finished:
                    for v in adj[u]:
                        need[v]-=1
                        if need[v]==0 and (not done[v]) and rem[v]>0:
                            enabled.append(v)
            t+=1
        dev_stats = {
            dids[i]: {"ticks_with_work": ticks_work[i], "ticks_with_multitask": ticks_mult[i]}
            for i in range(nD)
        }
        return t, dev_stats

    def scores(ch):
        ch=sanitize(ch)
        finish, dev_stats = simulate(ch)

        cm = (0.0 if MIN_TS<=0 or math.isinf(MIN_TS) else MIN_TS/max(finish,MIN_TS))
        ce_vals=[]
        for d in dids:
            tw=dev_stats[d]["ticks_with_work"]
            tm=dev_stats[d]["ticks_with_multitask"]
            ce_vals.append(1.0 if tw==0 else max(0.0,1.0-tm/tw))
        ce = sum(ce_vals)/len(ce_vals)

        total=ok=0
        for tid,t in tasks.items():
            for p in t.prereq:
                total+=1
                if set(ch[tid]) & set(ch[p]): ok+=1
        cc = ok/total if total else 1.0

        levels=sorted(set(d.level for d in devs)); dev_level={d.id:d.level for d in devs}; n=nT
        ba_vals=[]
        for lvl in levels:
            nums=[]
            for tid in tids:
                cnt=sum(1 for d in ch[tid] if dev_level.get(d)==lvl)
                nums.append(cnt*tasks[tid].effort)
            denom=sum(nums)
            if denom==0: ba_vals.append(1.0); continue
            ps=[x/denom for x in nums if x>0]
            H=0.0
            for p in ps: H -= p*(math.log(p)/math.log(n))
            ba_vals.append(H)
        ba=sum(ba_vals)/len(ba_vals) if ba_vals else 0.0

        fit = WEIGHTS["cm"]*cm + WEIGHTS["ce"]*ce + WEIGHTS["cc"]*cc + WEIGHTS["ba"]*ba
        return fit, {"cm":cm,"ce":ce,"cc":cc,"ba":ba,"sim":{"finish_time":finish, "dev_stats":dev_stats}}

    def init_pop():
        pop=[]
        for _ in range(pop_size):
            ch={tid: random.sample(dids, random.randint(1,max_per_task)) for tid in tids}
            pop.append(sanitize(ch))
        return pop

    def select(pop, fits):
        idxs=random.sample(range(len(pop)), tourn)
        b=max(idxs, key=lambda i: fits[i])
        return {tid: pop[b][tid][:] for tid in tids}

    def crossover(p1,p2):
        return {tid: (p1[tid][:] if random.random()<0.5 else p2[tid][:]) for tid in tids}

    def mutate(ch):
        for tid in tids:
            if random.random()<mut:
                op=random.choice(["add","remove","replace"])
                if op=="add" and len(ch[tid])<max_per_task:
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid].append(random.choice(cand))
                elif op=="remove" and len(ch[tid])>1:
                    ch[tid].pop(random.randrange(len(ch[tid])))
                elif op=="replace":
                    i=random.randrange(len(ch[tid]))
                    cand=[d for d in dids if d not in ch[tid]]
                    if cand: ch[tid][i]=random.choice(cand)
        return sanitize(ch)

    pop=init_pop()
    fits,dets=zip(*(scores(ch) for ch in pop))
    fits,dets=list(fits),list(dets)
    bi=max(range(len(pop)), key=lambda i: fits[i])
    best=(pop[bi], fits[bi], dets[bi])

    for g in range(generations):
        ranked=sorted(range(len(pop)), key=lambda i: fits[i], reverse=True)
        new_pop=[pop[i] for i in ranked[:elit]]
        new_fits=[fits[i] for i in ranked[:elit]]
        new_dets=[dets[i] for i in ranked[:elit]]
        while len(new_pop)<pop_size:
            c=mutate(crossover(select(pop,fits), select(pop,fits)))
            f,d=scores(c)
            new_pop.append(c); new_fits.append(f); new_dets.append(d)
        pop,fits,dets=new_pop,new_fits,new_dets
        bi=max(range(len(pop)), key=lambda i: fits[i])
        if fits[bi]>best[1]: best=(pop[bi], fits[bi], dets[bi])
        if verbose:
            print(f"Gen {g+1}/{generations} best fitness: {best[1]:.4f}")
    return best

def multitask_sum(best_details):
    return sum(v["ticks_with_multitask"] for v in best_details["sim"]["dev_stats"].values())

if __name__ == "__main__":
    tasks, devs = generate_set3_data()

    # ---- Run CaseAll
    W_ALL  = {"cm":0.25, "ce":0.25, "cc":0.25, "ba":0.25}
    chromA, fitA, detA = ga_set3(tasks, devs, W_ALL, verbose=True)
    timeA = detA["sim"]["finish_time"]
    multiA = multitask_sum(detA)

    # ---- Run CaseTime
    W_TIME = {"cm":1.0, "ce":0.0, "cc":0.0, "ba":0.0}
    chromT, fitT, detT = ga_set3(tasks, devs, W_TIME, verbose=True)
    timeT = detT["sim"]["finish_time"]
    multiT = multitask_sum(detT)

    # ---- Print table (exact format)
    print("=== COMPARISON TABLE ===")
    print(f"{'Set':<6} | {'Time(All)':>10} | {'Time(Time)':>10} | {'Multitask(All)':>15} | {'Multitask(Time)':>15}")
    print(f"{'Set3':<6} | {timeA:>10.2f} | {timeT:>10.2f} | {multiA:>15.2f} | {multiT:>15.2f}")

    # afficher la meilleure solution
    print("\n=== BEST SOLUTION (CaseAll) ===")
    print("Best fitness:", round(fitA, 6))
    print("Components:", {k: round(v, 6) for k, v in detA.items() if k in ["cm","ce","cc","ba"]})

    # afficher l'allocation des tâches (task -> devs)
    print("Finish time (sim):", detA["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromA, key=lambda x: int(x[1:])):  # t1, t2, ...
        print(f"  {tid} -> {chromA[tid]}")

    # afficher la meilleure solution pour CaseTime
    print("\n=== BEST SOLUTION (CaseTime) ===")
    print("Best fitness:", round(fitT, 6))
    print("Finish time (sim):", detT["sim"]["finish_time"])
    print("Assignments (task -> devs):")
    for tid in sorted(chromT, key=lambda x: int(x[1:])):
        print(f"  {tid} -> {chromT[tid]}")     

Gen 1/50 best fitness: 0.6973
Gen 2/50 best fitness: 0.7176
Gen 3/50 best fitness: 0.7176
Gen 4/50 best fitness: 0.7298
Gen 5/50 best fitness: 0.7458
Gen 6/50 best fitness: 0.7876
Gen 7/50 best fitness: 0.7918
Gen 8/50 best fitness: 0.8017
Gen 9/50 best fitness: 0.8076
Gen 10/50 best fitness: 0.8227
Gen 11/50 best fitness: 0.8242
Gen 12/50 best fitness: 0.8301
Gen 13/50 best fitness: 0.8478
Gen 14/50 best fitness: 0.8478
Gen 15/50 best fitness: 0.8686
Gen 16/50 best fitness: 0.8686
Gen 17/50 best fitness: 0.8686
Gen 18/50 best fitness: 0.8727
Gen 19/50 best fitness: 0.8727
Gen 20/50 best fitness: 0.8735
Gen 21/50 best fitness: 0.8762
Gen 22/50 best fitness: 0.8774
Gen 23/50 best fitness: 0.8783
Gen 24/50 best fitness: 0.8783
Gen 25/50 best fitness: 0.8783
Gen 26/50 best fitness: 0.8797
Gen 27/50 best fitness: 0.8797
Gen 28/50 best fitness: 0.8820
Gen 29/50 best fitness: 0.8820
Gen 30/50 best fitness: 0.8820
Gen 31/50 best fitness: 0.8852
Gen 32/50 best fitness: 0.8852
Gen 33/50 best fi


### Analysis and discussion

The following table presents the results obtained for the three input sets using our genetic algorithm implementation.  
For each set, two cases are evaluated:

- **CaseAll**: all objectives are considered simultaneously (time, multitasking, continuity, and balance).
- **CaseTime**: only the project completion time is optimized.

### Table : Comparison of experimental results obtained with this algorithm

| Metric | Set1 (11 tasks, 7 devs) |  | Set2 (11 tasks, 10 devs) |  | Set3 (21 tasks, 10 devs) |  |
|---|---|---|---|---|---|---|
|  | **All** | **Time** | **All** | **Time** | **All** | **Time** |
| **Project duration** | 824 | 833 | 749 | 949 | 981 | 965 |
| **Multitasking time** | 111 | 741 | 438 | 533 | 1106 | 1848 |
| **Mean (#devs / effortᵢ)** | 1.94e-02 | 3.05e-02 | 2.76e-02 | 3.62e-02 | 1.12e-02 | 1.96e-02 |
| **Variance (#devs / effortᵢ)** | 1.68e-04 | 3.91e-04 | 2.74e-04 | 4.18e-04 | 4.31e-05 | 8.87e-05 |
| **Best fitness** | 0.9488 | 1.0000 | 0.9475 | 1.0000 | 0.9098 | 1.0000 |


### Mean and Variance of (#Developers / Effort)

For each task \( i \), we compute the following ratio:
$$
r_i = \frac{|\text{Developers}_i|}{\text{Effort}_i}
$$
The **mean** value over all tasks is:
$$
\mu = \frac{1}{N} \sum_{i=1}^{N} r_i
$$
The **variance** is computed as:
$$
\sigma^2 = \frac{1}{N} \sum_{i=1}^{N} (r_i - \mu)^2
$$
where \( N \) is the total number of tasks.


#### Set1 (11 tasks, 7 developers)
In Set1, both cases lead to similar project durations (824 vs 833).  
However, CaseTime produces a very high multitasking time compared to CaseAll.  
This shows that optimizing only time leads to more parallel work and developer overload, while CaseAll keeps multitasking under control.

#### Set2 (11 tasks, 10 developers)
With more developers available, CaseAll achieves a shorter completion time than CaseTime (749 vs 949).  
This indicates that balancing objectives helps better distribute work across developers.  
Although multitasking is higher than in Set1, it remains much lower than in CaseTime.

#### Set3 (21 tasks, 10 developers)
Set3 is the most complex case due to the larger number of tasks.  
CaseAll results in a longer completion time than the smaller sets, which is expected.  
CaseTime slightly reduces the total duration but causes a very large increase in multitasking time, showing strong developer overload.

#### General observation
Across all sets, **CaseAll provides a better compromise** between project duration and work quality, while **CaseTime always reaches the best fitness for time but at the cost of excessive multitasking**.  
This confirms the importance of using a multi-objective approach when managing complex software projects.

For all three data sets, both the mean and the variance are higher in CaseTime than in CaseAll.
This indicates that when only project duration is optimized, more developers tend to be assigned to tasks,
especially to small-effort tasks.

The higher variance observed in CaseTime also shows that task assignments are less stable:
some small tasks receive too many developers, while large tasks may receive too few.
This behavior increases communication overhead and multitasking time.

In contrast, CaseAll produces more balanced and stable allocations by jointly considering
time, concentration, continuity, and balance objectives.